# SafeVision — PPE Detector Fine-tuning
**Nikhil Chaudhary | SafeVision · Autonex360 AI**

Fine-tuning YOLOv8n on the Ultralytics Construction-PPE dataset with a frozen backbone warm-up strategy.  
Four ablation runs: baseline / aug-only / freeze-only / aug-freeze (best expected).  
All metrics logged to WandB. Checkpoints saved every 5 epochs — Kaggle sessions die without warning.

**Runtime:** GPU T4 ×2 · ~5–7 hrs total for all 4 runs

In [ ]:
# Install deps — ultralytics bundles the dataset download so no manual steps needed
!pip install -q ultralytics wandb

In [ ]:
import os
import logging
import shutil
from pathlib import Path

import torch
import wandb
from ultralytics import YOLO

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger(__name__)

# Fail loud if no GPU — Kaggle occasionally starts sessions without one
# and silent CPU training would take 10x longer and time out
if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Go to Settings → Accelerator and enable GPU.')

n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    log.info(f'GPU {i}: {torch.cuda.get_device_name(i)}  '
             f'{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')
log.info(f'{n_gpu} GPU(s) available')


In [ ]:
# WandB login — key stored as Kaggle secret WANDB_API_KEY
# Add it: Notebook → Add-ons → Secrets → WANDB_API_KEY
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret('WANDB_API_KEY'))
log.info('WandB login ok')


In [ ]:
# ── shared config ─────────────────────────────────────────────────────────────

DATASET_YAML   = 'construction-ppe.yaml'   # auto-downloads on first use
BASE_WEIGHTS   = 'yolov8n.pt'
OUTPUT_DIR     = Path('/kaggle/working/safevision')
CKPT_DIR       = OUTPUT_DIR / 'checkpoints'
CKPT_INTERVAL  = 5       # save every N epochs — Kaggle kills sessions mid-run
IMG_SIZE       = 640
BATCH_SIZE     = 32      # T4 ×2: each GPU sees 16 images — DDP splits the batch
EPOCHS         = 100
WARMUP_EPOCHS  = 10      # frozen backbone phase
FREEZE_UNTIL   = 10      # layers 0–9 frozen during warm-up
DEVICE         = '0,1'  # both T4s; Ultralytics handles DDP internally

# Safety-critical classes — recall gate before promoting a checkpoint
RECALL_GATE = {'no-helmet': 0.80, 'no-safety-vest': 0.80}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

log.info(f'output dir: {OUTPUT_DIR}')
log.info(f'device: {DEVICE}  batch: {BATCH_SIZE} ({BATCH_SIZE // 2} per GPU)')


In [ ]:
# ── ablation configs ──────────────────────────────────────────────────────────
# Each run is defined by two flags. Everything else is held constant.
# I want to isolate the contribution of augmentation and backbone freezing
# independently before combining them.

ABLATIONS = [
    {
        'name':    'baseline',
        'aug':     False,
        'freeze':  False,
        'note':    'floor metric — no augmentation, full fine-tune from epoch 1',
    },
    {
        'name':    'aug-only',
        'aug':     True,
        'freeze':  False,
        'note':    'augmentation contribution in isolation',
    },
    {
        'name':    'freeze-only',
        'aug':     False,
        'freeze':  True,
        'note':    'backbone freeze contribution in isolation',
    },
    {
        'name':    'aug-freeze',
        'aug':     True,
        'freeze':  True,
        'note':    'expected best — both strategies combined',
    },
]

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────

def freeze_backbone(model, freeze_until):
    """Lock layers 0–(freeze_until-1). Head and neck stay trainable."""
    for i, (name, param) in enumerate(model.model.named_parameters()):
        param.requires_grad = i >= freeze_until
    frozen = sum(1 for p in model.model.parameters() if not p.requires_grad)
    log.info(f'frozen: {frozen} params (layers 0–{freeze_until - 1})')


def unfreeze_all(model):
    for p in model.model.parameters():
        p.requires_grad = True
    log.info('backbone unfrozen')


def make_train_args(cfg, phase_name, aug):
    """Build kwargs for model.train(). Aug flag toggles mosaic + hsv."""
    base = dict(
        data      = DATASET_YAML,
        imgsz     = IMG_SIZE,
        batch     = BATCH_SIZE,
        momentum  = 0.937,
        weight_decay = 5e-4,
        project   = str(OUTPUT_DIR),
        name      = phase_name,
        exist_ok  = True,
        verbose   = False,
        # don't log to WandB via Ultralytics — I handle callbacks manually
        # so I can log per-component loss rather than just final metrics
        plots     = False,
        device   = DEVICE,  # route to both T4s
    )
    if aug:
        base.update(dict(
            mosaic   = 1.0,   # highest-impact aug for small PPE at distance
            hsv_s    = 0.7,   # factory lighting varies a lot
            hsv_v    = 0.4,
            fliplr   = 0.5,
            flipud   = 0.0,   # workers don't hang upside down
            degrees  = 0.0,   # same reasoning
            mixup    = 0.1,
            copy_paste = 0.0, # creates contextually wrong scenes
        ))
    else:
        # disable all aug for baseline — mosaic=0 is the main one to kill
        base.update(dict(mosaic=0.0, hsv_s=0.0, hsv_v=0.0,
                         fliplr=0.0, mixup=0.0))
    return base


def check_quality_gate(model):
    """Evaluate on test split. Raises if safety-critical recall is below threshold."""
    results = model.val(data=DATASET_YAML, split='test', verbose=False)
    per_class = {}
    for idx, class_idx in enumerate(results.box.ap_class_index):
        name = results.names[int(class_idx)]
        per_class[name] = {
            'precision': float(results.box.p[idx]),
            'recall':    float(results.box.r[idx]),
            'ap50':      float(results.box.ap50[idx]),
        }
    failed = []
    for cls, min_r in RECALL_GATE.items():
        if cls in per_class and per_class[cls]['recall'] < min_r:
            failed.append(f"{cls} recall={per_class[cls]['recall']:.3f} < {min_r}")
    if failed:
        raise AssertionError('Quality gate failed: ' + ', '.join(failed))
    return per_class


In [ ]:
# ── main training function ────────────────────────────────────────────────────

def run_ablation(cfg):
    name   = cfg['name']
    aug    = cfg['aug']
    freeze = cfg['freeze']

    log.info(f'--- starting run: {name} (aug={aug}, freeze={freeze}) ---')

    wandb_run = wandb.init(
        project = 'safevision-ppe',
        name    = name,
        config  = {
            'model':           BASE_WEIGHTS,
            'dataset':         DATASET_YAML,
            'epochs':          EPOCHS,
            'warmup_epochs':   WARMUP_EPOCHS if freeze else 0,
            'freeze_until':    FREEZE_UNTIL if freeze else 0,
            'augmentation':    aug,
            'img_size':        IMG_SIZE,
            'batch_size':      BATCH_SIZE,
            'note':            cfg['note'],
            # ultralytics version as dataset proxy — construction-ppe.yaml
            # doesn't have its own version field
            'ultralytics_ver': __import__('ultralytics').__version__,
        },
        reinit = True,
    )

    model = YOLO(BASE_WEIGHTS)

    # ── per-epoch WandB callback ───────────────────────────────────────────────
    ckpt_counter = {'n': 0}  # closure — track epoch count across both training phases

    def on_train_epoch_end(trainer):
        ckpt_counter['n'] += 1
        ep = ckpt_counter['n']
        m  = trainer.metrics or {}

        # Log loss components separately — if cls_loss dominates in early epochs
        # it signals class imbalance. Conflating all losses hides this.
        log_dict = {'epoch': ep}
        if trainer.loss_items is not None:
            log_dict.update({
                'train/box_loss': float(trainer.loss_items[0]),
                'train/cls_loss': float(trainer.loss_items[1]),
                'train/dfl_loss': float(trainer.loss_items[2]),
            })
        log_dict.update({
            'metrics/mAP50':     m.get('metrics/mAP50(B)'),
            'metrics/mAP50-95':  m.get('metrics/mAP50-95(B)'),
            'metrics/precision': m.get('metrics/precision(B)'),
            'metrics/recall':    m.get('metrics/recall(B)'),
            'lr':                trainer.optimizer.param_groups[0]['lr'],
        })
        wandb_run.log(log_dict)

        # Interval checkpoint — Kaggle sessions die without warning
        if ep % CKPT_INTERVAL == 0:
            ckpt_path = CKPT_DIR / f'{name}_epoch{ep:04d}.pt'
            # DDP wraps model.model in .module — unwrap before saving
            raw = model.model.module if hasattr(model.model, 'module') else model.model
            torch.save(raw.state_dict(), ckpt_path)
            log.info(f'checkpoint: {ckpt_path.name}')

    model.add_callback('on_train_epoch_end', on_train_epoch_end)

    # ── phase A: warm-up (frozen backbone) or full train if freeze=False ───────
    if freeze:
        log.info(f'warm-up phase: {WARMUP_EPOCHS} epochs, backbone frozen')
        freeze_backbone(model, FREEZE_UNTIL)
        model.train(
            epochs = WARMUP_EPOCHS,
            lr0    = 0.01,
            lrf    = 0.01,
            **make_train_args(cfg, f'{name}_warmup', aug),
        )
        # Save warm-up state before unfreezing — useful if full run OOMs
        warmup_path = CKPT_DIR / f'{name}_after_warmup.pt'
        raw = model.model.module if hasattr(model.model, 'module') else model.model
        torch.save(raw.state_dict(), warmup_path)
        unfreeze_all(model)
        remaining = EPOCHS - WARMUP_EPOCHS
        # Lower LR for the unfrozen phase — backbone was frozen until now
        # so a large LR would destabilise the COCO features we just preserved
        lr_main = 0.001
    else:
        remaining = EPOCHS
        lr_main   = 0.01

    # ── phase B: full fine-tune ────────────────────────────────────────────────
    # Reusing the same model object after DDP training corrupts internal state —
    # KeyError('model') on the second .train() call. Reload from the warmup
    # checkpoint Ultralytics already saved in proper format.
    if freeze:
        warmup_best = RUN_DIR / f'{name}_warmup' / 'weights' / 'best.pt'
        model = YOLO(str(warmup_best))
        model.add_callback('on_train_epoch_end', on_train_epoch_end)
    log.info(f'full fine-tune: {remaining} epochs')
    model.train(
        epochs = remaining,
        lr0    = lr_main,
        lrf    = 0.01,
        **make_train_args(cfg, name, aug),
    )

    # ── quality gate ──────────────────────────────────────────────────────────
    log.info('running quality gate...')
    try:
        per_class = check_quality_gate(model)
        wandb_run.log({'quality_gate': 'passed'})
        for cls_name, m in per_class.items():
            wandb_run.log({
                f'test/{cls_name}/recall':    m['recall'],
                f'test/{cls_name}/precision': m['precision'],
                f'test/{cls_name}/ap50':      m['ap50'],
            })
    except AssertionError as e:
        log.warning(f'quality gate failed: {e}')
        wandb_run.log({'quality_gate': 'failed', 'gate_reason': str(e)})

    # Save final weights
    final_path = CKPT_DIR / f'{name}_final.pt'
    model.save(str(final_path))
    log.info(f'saved: {final_path}')

    # Log as WandB artifact so it persists after session ends
    artifact = wandb.Artifact(f'ppe-detector-{name}', type='model')
    artifact.add_file(str(final_path))
    wandb_run.log_artifact(artifact)

    wandb_run.finish()
    log.info(f'--- run {name} complete ---')
    return final_path

In [ ]:
# ── run ablations ─────────────────────────────────────────────────────────────
# Run them in this order. Baseline first — if setup is broken you find out in
# 1.5 hrs, not 6. While this cell runs, open a second Kaggle session and
# start aug-only there in parallel.

# To run a single ablation (e.g. just aug-freeze), change the list:
#   run_ablation(ABLATIONS[3])  # 0=baseline, 1=aug-only, 2=freeze-only, 3=aug-freeze

results_map = {}

for cfg in ABLATIONS:
    try:
        final_path = run_ablation(cfg)
        results_map[cfg['name']] = str(final_path)
    except Exception as e:
        # Don't abort all runs if one fails — log and continue
        log.error(f"run '{cfg['name']}' failed: {e}")
        results_map[cfg['name']] = f'FAILED: {e}'

print('\n=== all runs summary ===')
for name, path in results_map.items():
    print(f'  {name:15s}  {path}')

In [ ]:
# ── evaluate best run (aug-freeze) ────────────────────────────────────────────
# Run this cell separately after all ablations complete.
# Prints the per-class recall table you'll copy into the README.

best_weights = CKPT_DIR / 'aug-freeze_final.pt'

if not best_weights.exists():
    print(f'weights not found at {best_weights} — check CKPT_DIR listing below')
    import os
    for f in sorted(CKPT_DIR.iterdir()):
        print(f'  {f.name}')
else:
    model = YOLO(str(best_weights))
    results = model.val(data=DATASET_YAML, split='test', verbose=False)

    print(f"\nmAP50:    {results.box.map50:.4f}")
    print(f"mAP50-95: {results.box.map:.4f}")
    print(f"Precision: {results.box.mp:.4f}")
    print(f"Recall:    {results.box.mr:.4f}")

    print(f"\n{'class':<22}  {'P':>6}  {'R':>6}  {'AP50':>6}")
    print('-' * 42)
    for idx, class_idx in enumerate(results.box.ap_class_index):
        name = results.names[int(class_idx)]
        gate = ' ← gate' if name in RECALL_GATE else ''
        print(f"{name:<22}  {results.box.p[idx]:>6.3f}  "
              f"{results.box.r[idx]:>6.3f}  {results.box.ap50[idx]:>6.3f}{gate}")

In [ ]:
# ── download best weights ─────────────────────────────────────────────────────
# Kaggle sessions expire. Run this cell to make weights available as a
# Kaggle dataset output that persists.
#
# Alternatively: go to Output panel (right side) → download aug-freeze_final.pt

best = CKPT_DIR / 'aug-freeze_final.pt'
dest = Path('/kaggle/working/best.pt')

if best.exists():
    shutil.copy(best, dest)
    print(f'weights ready to download at: {dest}')
    print(f'size: {dest.stat().st_size / 1e6:.1f} MB')
else:
    print('aug-freeze_final.pt not found — check run completed successfully')